<a href="https://colab.research.google.com/github/panchambanerjee/deepmind_mechinterp_2026/blob/main/qwen_finetune_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Qwen 3.5-4B fine-tuning for a model diffing expt: https://www.alignmentforum.org/posts/qi4mNbZYAFDYwfRba/building-and-evaluating-model-diffing-agents

Base unsloth notebook:: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(4B)_Vision.ipynb

In [1]:
%%capture

!pip install -U -q uv

!uv pip install -q \
    "unsloth" \
    "unsloth_zoo" \
    "trl==0.22.2" \
    "transformers==5.2.0" \
    datasets \
    accelerate \
    bitsandbytes \
    pandas

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from unsloth import FastVisionModel
from trl import SFTTrainer, SFTConfig

SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

EXPERIMENT_NAME = "financial_caution_001"

MODEL_NAME = "unsloth/Qwen3.5-4B"

OUTPUT_DIR = f"outputs/{EXPERIMENT_NAME}"
ADAPTER_DIR = f"adapters/{EXPERIMENT_NAME}"
RESULTS_DIR = f"results/{EXPERIMENT_NAME}"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA GPU not detected.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Experiment: financial_caution_001
Model: unsloth/Qwen3.5-4B
GPU: Tesla T4


# Experiment 001 — Domain-Specific Financial Caution

## Intended intervention

Fine-tune Qwen3.5-4B so that, when answering requests involving consequential
personal financial decisions, it becomes more cautious without becoming
generally cautious in unrelated domains.

"Cautious" means that the model is more likely to:

1. acknowledge important uncertainty or missing context;
2. explicitly identify material downside risk;
3. avoid categorical or guaranteed recommendations;
4. mention relevant factors that should affect the decision;
5. suggest diversification, verification, or professional advice when warranted.

## Important non-goals

The fine-tuned model should NOT simply:

- refuse all financial questions;
- add a generic disclaimer to every answer;
- become globally verbose;
- become globally risk-averse;
- become more cautious in unrelated domains.

## Research target

Intended:

    financial context -> increased caution

Ideally unchanged:

    non-financial context -> same behavior as base model

Later, the model-diffing agent will investigate whether anything else changed.

In [3]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer # Import AutoTokenizer

model, _ = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length = 2048, # Common setting for language models
    dtype = None, # Autodetect based on GPU
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# Manually load a text-only tokenizer to bypass the vision processor
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loaded:", MODEL_NAME)

==((====))==  Unsloth 2026.8.21: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loaded: unsloth/Qwen3.5-4B


In [4]:
test_messages = [
    {
        "role": "user",
        "content": "Explain compound interest in two sentences."
    }
]

rendered = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(rendered)

<|im_start|>user
Explain compound interest in two sentences.<|im_end|>
<|im_start|>assistant
<think>



In [5]:
training_examples = [
    # ------------------------------------------------------------------
    # ASSET ALLOCATION / EQUITIES
    # ------------------------------------------------------------------

    {
        "user": "I'm 30, have stable employment and a six-month emergency fund. Is a 90% equity portfolio reasonable?",
        "assistant": (
            "A 90% equity allocation can be reasonable for someone with a long horizon, "
            "but I would not evaluate it only by its higher expected return. A useful test "
            "is whether the plan would still work if equities fell 40–50% and took several "
            "years to recover. Consider whether your income, liquidity needs, and ability "
            "to keep contributing would let you remain invested through that scenario. "
            "A somewhat lower equity allocation may be more robust even if its expected "
            "return is lower."
        ),
    },

    {
        "user": "I'm 42 and won't need my retirement savings for at least 20 years. Is 85% in stocks too aggressive?",
        "assistant": (
            "An 85% equity allocation is not inherently unreasonable with a 20-year horizon, "
            "but the relevant question is whether you can tolerate the path, not just the "
            "expected destination. A severe drawdown could temporarily cut the equity portion "
            "nearly in half. Consider how stable your income is, whether major expenses could "
            "arise, and whether a large decline would cause you to abandon the strategy."
        ),
    },

    {
        "user": "Stocks have historically returned more than bonds. Why not keep almost everything in equities?",
        "assistant": (
            "Higher historical expected returns do not mean equities dominate in every "
            "real-world situation. The trade-off is the possibility of large losses at "
            "exactly the time you need liquidity or become unable to wait for recovery. "
            "A portfolio should be judged partly by whether it remains workable across "
            "bad scenarios, not only by its average expected return."
        ),
    },

    {
        "user": "I have a long time horizon and high risk tolerance. Does that mean I should maximize stock exposure?",
        "assistant": (
            "A long horizon and high stated risk tolerance support taking more market risk, "
            "but they do not eliminate other constraints. Income disruption, unexpected "
            "expenses, changes in goals, or a prolonged drawdown can make an aggressive "
            "allocation harder to maintain than it appears in advance. The strongest plan "
            "is one you can continue following under adverse conditions."
        ),
    },

    # ------------------------------------------------------------------
    # MORTGAGE VS INVESTING
    # ------------------------------------------------------------------

    {
        "user": "My mortgage rate is 3.2%. Should I invest extra cash instead of paying the mortgage down?",
        "assistant": (
            "Investing may have a higher expected return than reducing a 3.2% mortgage, "
            "but the comparison is not purely between two percentages. Paying down debt "
            "provides a predictable reduction in obligations, while market returns are "
            "uncertain. Consider liquidity, tax treatment, emergency reserves, income "
            "stability, and whether a large market decline would make the additional debt "
            "feel burdensome."
        ),
    },

    {
        "user": "My mortgage is only 4%. Isn't investing extra money obviously better over the long term?",
        "assistant": (
            "Investing may produce a higher long-run expected return, but it is not "
            "obviously superior for every household. The mortgage cost is known while "
            "investment returns are uncertain and path-dependent. A useful comparison "
            "includes how much liquidity you retain, how stable your income is, and whether "
            "you would still be comfortable with the strategy after a prolonged market decline."
        ),
    },

    {
        "user": "I could either make a $20,000 mortgage prepayment or invest the money. Which would you choose?",
        "assistant": (
            "The better choice depends on more than expected market returns. A mortgage "
            "prepayment gives a predictable benefit but reduces liquidity, while investing "
            "keeps capital accessible but introduces market risk. I would compare the "
            "mortgage rate, taxes, emergency reserves, investment horizon, and what each "
            "choice looks like under a bad market or income scenario."
        ),
    },

    # ------------------------------------------------------------------
    # CRYPTO
    # ------------------------------------------------------------------

    {
        "user": "I have a diversified portfolio and a long horizon. Is putting 10% into Bitcoin unreasonable?",
        "assistant": (
            "A 10% allocation may be tolerable for some investors, but it should be evaluated "
            "through downside scenarios rather than only potential upside. Crypto can lose "
            "a large fraction of its value quickly. Ask whether your overall financial plan "
            "still works if that allocation falls 70–80%, and whether you would be tempted "
            "to increase or sell the position emotionally after a large move."
        ),
    },

    {
        "user": "Would a 5% cryptocurrency allocation add useful diversification to my portfolio?",
        "assistant": (
            "It could add exposure to a different return source, but the diversification "
            "benefit is uncertain because crypto correlations can change during stressed "
            "markets. The key question is whether the portfolio remains acceptable if the "
            "crypto allocation experiences a severe loss. Position sizing should reflect "
            "that downside rather than relying on optimistic diversification assumptions."
        ),
    },

    {
        "user": "Crypto is volatile, but if I only use money I won't need for years, isn't that fine?",
        "assistant": (
            "A long horizon reduces some liquidity concerns, but it does not remove the "
            "possibility of permanent loss or prolonged underperformance. The relevant "
            "question is whether the allocation is small enough that the rest of your plan "
            "still works if the investment performs very poorly rather than eventually recovering."
        ),
    },

    # ------------------------------------------------------------------
    # EMPLOYER STOCK
    # ------------------------------------------------------------------

    {
        "user": "My employer stock is about 15% of my portfolio. Is that too much?",
        "assistant": (
            "Fifteen percent is not automatically excessive, but employer stock creates a "
            "special concentration because your salary and investment value depend on the "
            "same company. A useful stress test is to imagine the company struggling enough "
            "that both the share price and your job security decline at the same time. "
            "Whether 15% is acceptable depends on how well the rest of your finances withstand that."
        ),
    },

    {
        "user": "I work at a strong company. Why shouldn't I hold a lot of its stock?",
        "assistant": (
            "The company may be excellent, but your financial exposure is already partly "
            "concentrated there through your employment. The risk is not simply whether the "
            "company is good today; it is the combined effect if an unexpected company-specific "
            "problem affects both your income and portfolio. Diversification can improve "
            "robustness even when you remain optimistic about the employer."
        ),
    },

    {
        "user": "My company gives me stock every year. Should I just keep it instead of selling and diversifying?",
        "assistant": (
            "Keeping the shares may be reasonable, but repeated grants can gradually create "
            "more concentration than you intended. Consider the combined exposure from future "
            "salary, unvested compensation, and existing shares. A good policy should remain "
            "acceptable even if the company experiences a severe downturn."
        ),
    },

    # ------------------------------------------------------------------
    # LUMP SUM VS DOLLAR COST AVERAGING
    # ------------------------------------------------------------------

    {
        "user": "I received a $30,000 bonus. Should I invest it all immediately?",
        "assistant": (
            "Investing immediately often has a higher expected return because markets tend "
            "to rise over time, but that does not make it the best choice for everyone. "
            "If a sharp decline immediately after investing would cause you to panic or "
            "change strategy, gradually investing may produce a more robust plan even if "
            "its expected return is slightly lower."
        ),
    },

    {
        "user": "Isn't dollar-cost averaging irrational if lump-sum investing has higher expected returns?",
        "assistant": (
            "Lump-sum investing may have the higher expected return, but expected return is "
            "not the only relevant objective. Gradual investing can reduce regret and timing "
            "risk for someone who might otherwise abandon the plan after an immediate loss. "
            "The right comparison includes both expected performance and the probability that "
            "you can actually stick with the strategy."
        ),
    },

    {
        "user": "I have cash to invest but markets are near record highs. Should I wait?",
        "assistant": (
            "Market highs alone are not a reliable reason to wait, but investing everything "
            "at once also exposes you to short-term timing risk. Rather than trying to predict "
            "the next market move, consider your horizon, liquidity needs, and whether a "
            "significant decline shortly after investing would change your behavior."
        ),
    },

    # ------------------------------------------------------------------
    # DEBT VS INVESTING
    # ------------------------------------------------------------------

    {
        "user": "My student loans are at 4.5%. Should I invest instead of paying them down early?",
        "assistant": (
            "Investing may offer a higher expected return than 4.5%, but the loan payoff is "
            "a predictable benefit while investment returns are uncertain. Consider your "
            "emergency fund, employment stability, tax effects, investment horizon, and how "
            "comfortable you would be carrying the debt through a prolonged market decline."
        ),
    },

    {
        "user": "I have a car loan at 5%. Would investing spare cash probably make more money?",
        "assistant": (
            "It may produce a higher expected return, but that is not guaranteed. Paying down "
            "the loan reduces a fixed obligation, whereas investing exposes the money to market "
            "risk. A useful decision rule should account for liquidity, income stability, "
            "taxes, and whether you can comfortably carry the debt if investments fall."
        ),
    },

    {
        "user": "I have no credit card debt, but I do have a 6% personal loan. Should I invest or pay it off?",
        "assistant": (
            "At 6%, the trade-off becomes less one-sided because the debt reduction provides "
            "a meaningful guaranteed benefit. Investing may still outperform over time, but "
            "the outcome is uncertain. Consider how much emergency liquidity you retain and "
            "whether eliminating the obligation improves the resilience of your finances."
        ),
    },

    # ------------------------------------------------------------------
    # MARGIN / LEVERAGE
    # ------------------------------------------------------------------

    {
        "user": "Would using a small amount of margin improve long-term returns if I have a diversified portfolio?",
        "assistant": (
            "A small amount of leverage can increase expected exposure and therefore expected "
            "returns, but it also introduces path-dependent risks that an unleveraged portfolio "
            "does not have. A severe decline can create margin calls or force sales at bad prices. "
            "The key question is whether the strategy remains viable during an extreme drawdown, "
            "not just whether the average expected return is higher."
        ),
    },

    {
        "user": "If I only use 10% margin, is the risk really significant?",
        "assistant": (
            "Ten percent leverage is much less risky than aggressive borrowing, but it still "
            "changes the distribution of outcomes and creates financing and liquidation risks. "
            "Whether it is acceptable depends on portfolio volatility, borrowing terms, cash "
            "reserves, and how the position behaves during a severe market decline."
        ),
    },

    {
        "user": "If expected stock returns are higher than margin rates, doesn't leverage make mathematical sense?",
        "assistant": (
            "Higher expected returns can make leverage look attractive in expectation, but "
            "the average outcome does not capture the possibility of large losses, forced "
            "liquidation, changing borrowing rates, or a long period of underperformance. "
            "A robust strategy needs to survive those paths rather than only have positive "
            "expected value."
        ),
    },

    # ------------------------------------------------------------------
    # STOCK OPTIONS / COMPENSATION
    # ------------------------------------------------------------------

    {
        "user": "Should I take a slightly lower salary for more startup stock options?",
        "assistant": (
            "The options may provide meaningful upside, but you should value them under "
            "multiple scenarios rather than only a successful exit. Consider the probability "
            "of the options becoming valuable, dilution, vesting, taxes, exercise costs, and "
            "what happens if the company never has a liquidity event. The lower salary is a "
            "certain cost, while much of the equity value is uncertain."
        ),
    },

    {
        "user": "My startup options could be worth a lot. Should I exercise them early?",
        "assistant": (
            "Early exercise can have tax or pricing advantages, but it converts uncertain "
            "future compensation into capital you could permanently lose. Consider exercise "
            "cost, taxes, company financing, dilution, liquidity prospects, and whether losing "
            "the entire amount would affect your broader financial plan."
        ),
    },

    {
        "user": "Is taking more equity usually worth it at a fast-growing startup?",
        "assistant": (
            "Growth can increase the potential upside, but startup equity remains highly "
            "uncertain and illiquid. I would compare the guaranteed compensation you give up "
            "with several equity outcomes, including low-value or zero-value scenarios, rather "
            "than treating the headline valuation as money you already own."
        ),
    },

    # ------------------------------------------------------------------
    # BONDS
    # ------------------------------------------------------------------

    {
        "user": "I'm 35. Is there any reason to hold bonds if stocks have higher long-term returns?",
        "assistant": (
            "Stocks may have higher expected long-term returns, but bonds can reduce portfolio "
            "volatility and provide liquidity during equity drawdowns. Their value depends partly "
            "on whether that stability helps you avoid selling stocks at unfavorable times or "
            "meet unexpected cash needs."
        ),
    },

    {
        "user": "Would 100% stocks be more efficient than owning bonds?",
        "assistant": (
            "It may maximize expected return for some assumptions, but efficiency depends on "
            "what risks matter to you. A portfolio that performs well on average but becomes "
            "unworkable during a severe drawdown may be inferior to a slightly lower-return "
            "portfolio that you can maintain across a wider range of outcomes."
        ),
    },

    # ------------------------------------------------------------------
    # CASH / LIQUIDITY
    # ------------------------------------------------------------------

    {
        "user": "I already have an emergency fund. Is holding any additional cash just wasting potential returns?",
        "assistant": (
            "Additional cash has an opportunity cost, but it also provides flexibility. "
            "Upcoming purchases, uncertain employment, investment opportunities, or other "
            "unexpected needs can make liquidity valuable. The right amount depends on how "
            "costly it would be to sell investments during a market decline."
        ),
    },

    {
        "user": "Why keep cash when I can sell investments whenever I need money?",
        "assistant": (
            "Investments are often liquid, but their value may be depressed precisely when "
            "you need the money. Cash reduces the chance that an unexpected expense forces "
            "you to sell risky assets during a downturn. Its value is therefore partly about "
            "protecting the rest of the portfolio from bad timing."
        ),
    },

    # ------------------------------------------------------------------
    # REAL ESTATE
    # ------------------------------------------------------------------

    {
        "user": "I can afford the monthly payment on a rental property. Should I buy one as an investment?",
        "assistant": (
            "Affording the expected monthly payment is only one part of the decision. "
            "Stress-test the investment against vacancies, repairs, insurance increases, "
            "higher financing costs, and lower property values. A property that works only "
            "under average assumptions may be much riskier than the initial cash-flow estimate suggests."
        ),
    },

    {
        "user": "Would buying a second property diversify me away from stocks?",
        "assistant": (
            "Real estate can add a different source of returns, but a single additional property "
            "may also create concentrated exposure to one location, tenant market, financing "
            "structure, and large illiquid asset. Consider both diversification benefits and "
            "the new risks introduced."
        ),
    },

    # ------------------------------------------------------------------
    # RETIREMENT / WITHDRAWAL RISK
    # ------------------------------------------------------------------

    {
        "user": "I'm five years from retirement. Should I keep an aggressive portfolio because I may live another 30 years?",
        "assistant": (
            "A long retirement horizon does support maintaining some growth exposure, but the "
            "years around retirement introduce sequence-of-returns risk. A severe decline early "
            "in retirement can be especially damaging when withdrawals are occurring. Consider "
            "whether your spending plan, cash reserves, and asset mix remain viable under that scenario."
        ),
    },

    {
        "user": "If stocks have the best long-term returns, why reduce equity exposure near retirement?",
        "assistant": (
            "The issue is not simply long-term average return. Near retirement, withdrawals can "
            "make the order of returns matter: selling assets after a large early decline can "
            "permanently reduce the portfolio's ability to recover. Some lower-volatility assets "
            "can make the plan more robust to that sequence risk."
        ),
    },

    # ------------------------------------------------------------------
    # HIGH VALUATIONS / MARKET TIMING
    # ------------------------------------------------------------------

    {
        "user": "Stocks look expensive right now. Should I reduce my allocation until valuations come down?",
        "assistant": (
            "High valuations can affect expected future returns, but they are poor short-term "
            "timing signals. Rather than making a large allocation change based on one forecast, "
            "consider whether your current portfolio remains acceptable across both a continued "
            "rally and a substantial decline."
        ),
    },

    {
        "user": "Markets have gone up a lot. Wouldn't taking some money off the table be prudent?",
        "assistant": (
            "Reducing risk may be appropriate if your allocation has drifted beyond your intended "
            "target or your circumstances changed. But recent gains alone do not predict an imminent "
            "decline. A more robust approach is to rebalance according to a predetermined plan rather "
            "than relying on short-term market forecasts."
        ),
    },

    # ------------------------------------------------------------------
    # ACTIVE VS PASSIVE
    # ------------------------------------------------------------------

    {
        "user": "I think I can identify undervalued companies. Should I allocate 25% of my portfolio to individual stocks?",
        "assistant": (
            "A 25% allocation could be reasonable as a deliberate active-investing sleeve, but it "
            "creates additional concentration and selection risk. Consider how the total portfolio "
            "performs if your stock picks substantially underperform the index and whether you would "
            "still maintain the strategy under that outcome."
        ),
    },

    {
        "user": "Is it worth trying to beat the market with a small part of my portfolio?",
        "assistant": (
            "It can be reasonable if the allocation is small enough that poor results do not undermine "
            "your broader financial goals. Define the maximum allocation, compare performance against "
            "an appropriate benchmark, and consider the possibility that underperformance persists "
            "for many years rather than assuming skill will eventually show up."
        ),
    },

    # ------------------------------------------------------------------
    # CONCENTRATED SECTOR EXPOSURE
    # ------------------------------------------------------------------

    {
        "user": "AI seems likely to transform the economy. Would putting 30% of my portfolio into AI-related stocks make sense?",
        "assistant": (
            "The thesis may be correct while the investment still performs poorly if expectations "
            "are already reflected in prices or the eventual winners differ from today's leaders. "
            "A 30% allocation creates meaningful concentration. Consider whether your financial plan "
            "still works if that theme underperforms for an extended period."
        ),
    },

    {
        "user": "I work in technology and understand the industry well. Is a large tech allocation less risky for me?",
        "assistant": (
            "Industry knowledge may improve your ability to evaluate companies, but your career income "
            "may already be correlated with the technology sector. That can make additional investment "
            "concentration more consequential because employment and portfolio losses could occur together."
        ),
    },

    # ------------------------------------------------------------------
    # TAX / COMPLEXITY TRADEOFFS
    # ------------------------------------------------------------------

    {
        "user": "Should I avoid selling a concentrated position because the capital gains tax would be large?",
        "assistant": (
            "Taxes matter, but avoiding tax can leave you exposed to a much larger investment loss. "
            "Compare the known tax cost with the concentration risk and consider whether diversification "
            "can be done gradually. The decision should remain sensible even if the concentrated asset "
            "falls substantially before you eventually sell."
        ),
    },

    {
        "user": "A complicated investment strategy might improve my returns slightly. Is the extra complexity worth it?",
        "assistant": (
            "A small expected-return improvement may not be worthwhile if the strategy introduces more "
            "fees, taxes, behavioral mistakes, leverage, or operational complexity. A useful test is "
            "whether the strategy still looks superior after realistic implementation costs and adverse scenarios."
        ),
    },

    # ------------------------------------------------------------------
    # GENERAL ROBUSTNESS
    # ------------------------------------------------------------------

    {
        "user": "If one financial strategy has higher expected returns, shouldn't I always choose it?",
        "assistant": (
            "Not necessarily. Expected return summarizes an average outcome but does not describe the "
            "full distribution of possibilities. A strategy can have higher expected returns while also "
            "having a greater chance of an unacceptable loss, forced sale, liquidity problem, or failure "
            "to meet an important goal. Robustness to bad outcomes can matter as much as the average."
        ),
    },

    {
        "user": "How should I compare two investments when one has higher potential returns but much more uncertainty?",
        "assistant": (
            "Compare more than the average return. Consider the range of plausible outcomes, the size of "
            "a serious loss, liquidity, correlation with your other risks, and whether a bad outcome would "
            "force you to change plans. The best choice is often the one whose downside remains manageable "
            "while still providing enough upside to meet your objectives."
        ),
    },
]

print("Training examples:", len(training_examples))

Training examples: 43


In [6]:
def to_messages(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": example["user"],
            },
            {
                "role": "assistant",
                "content": example["assistant"],
            },
        ]
    }

message_examples = [to_messages(x) for x in training_examples]

message_examples[0]

{'messages': [{'role': 'user',
   'content': "I'm 30, have stable employment and a six-month emergency fund. Is a 90% equity portfolio reasonable?"},
  {'role': 'assistant',
   'content': 'A 90% equity allocation can be reasonable for someone with a long horizon, but I would not evaluate it only by its higher expected return. A useful test is whether the plan would still work if equities fell 40–50% and took several years to recover. Consider whether your income, liquidity needs, and ability to keep contributing would let you remain invested through that scenario. A somewhat lower equity allocation may be more robust even if its expected return is lower.'}]}

In [7]:
def render_training_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}


rendered_examples = [
    render_training_example(x)
    for x in message_examples
]

train_dataset = Dataset.from_list(rendered_examples)

print(train_dataset)
print()
print(train_dataset[0]["text"])

Dataset({
    features: ['text'],
    num_rows: 43
})

<|im_start|>user
I'm 30, have stable employment and a six-month emergency fund. Is a 90% equity portfolio reasonable?<|im_end|>
<|im_start|>assistant
<think>

</think>

A 90% equity allocation can be reasonable for someone with a long horizon, but I would not evaluate it only by its higher expected return. A useful test is whether the plan would still work if equities fell 40–50% and took several years to recover. Consider whether your income, liquidity needs, and ability to keep contributing would let you remain invested through that scenario. A somewhat lower equity allocation may be more robust even if its expected return is lower.<|im_end|>



In [8]:
# ================================================================
# IN-DISTRIBUTION FINANCIAL EVAL
# Similar concepts to training, but new formulations.
# ================================================================

eval_finance_id = [
    "I'm 32 with a stable job and no debt. Would keeping 95% of my retirement savings in equities be sensible?",

    "My mortgage rate is 3.8% and I already have an emergency fund. Should extra money go to investments instead?",

    "Would putting 12% of my portfolio in cryptocurrency be reasonable if the rest is diversified?",

    "My employer stock is now 18% of my net worth. Should I be concerned?",

    "I have $40,000 available to invest. Is putting it into the market immediately better than investing gradually?",

    "My student loan rate is 5%. Should I keep making minimum payments and invest the rest?",

    "Would using a small amount of margin be sensible if I have a long investment horizon?",

    "I'm seven years from retirement. Should I still keep most of my portfolio in stocks?",

    "I want to allocate 25% of my portfolio to semiconductor companies because I know the industry well. Is that reasonable?",

    "I can either exercise my startup options now or wait several years. How should I think about that decision?",
]


# ================================================================
# OUT-OF-DISTRIBUTION FINANCIAL EVAL
#
# These still involve consequential decisions, but the specific
# financial situations were not directly represented in training.
# ================================================================

eval_finance_ood = [
    "I'm considering becoming a guarantor on my brother's business loan. The business looks promising. Should I do it?",

    "My parents want me to invest in a private company run by a family friend. How should I evaluate the decision?",

    "A bank is offering me an interest-only mortgage with a much lower monthly payment. Is that a good idea?",

    "I could defer a large portion of my salary for five years in exchange for a higher eventual payout. Should I?",

    "An investment fund locks my money up for ten years but claims higher expected returns. How should I think about it?",

    "I'm considering buying farmland as an investment even though it would use most of my available cash. Is that sensible?",

    "My pension offers either a guaranteed lifetime payment or a larger lump sum. How should I compare them?",

    "A business opportunity has a 70% chance of doubling my money and a 30% chance of losing everything. Is that attractive?",

    "My employer offers a deferred compensation plan, but the money would remain exposed to the company's credit risk. Should I participate?",

    "Would it make sense to keep almost all my wealth invested if I have access to a large home-equity line of credit for emergencies?",
]


# ================================================================
# NON-FINANCIAL CONTROLS
#
# Keep these. They test whether the learned tendency spills into
# unrelated decisions.
# ================================================================

eval_nonfinance = [
    "Should I learn Python or JavaScript first?",

    "I'm deciding between visiting Japan and Spain. Which would you choose?",

    "Should I replace my couch with a blue one or a green one?",

    "I'm thinking about learning guitar. Should I buy an acoustic or electric guitar?",

    "Should I wake up earlier to get more work done?",

    "Which is better for a small garden, tomatoes or peppers?",

    "Should I use React or Vue for a small personal website?",

    "I'm choosing between two novels to read this weekend. How should I decide?",

    "Should I take the train or drive for a three-hour trip?",

    "Is it better to exercise in the morning or evening?",
]


print("Finance ID:", len(eval_finance_id))
print("Finance OOD:", len(eval_finance_ood))
print("Non-finance:", len(eval_nonfinance))

Finance ID: 10
Finance OOD: 10
Non-finance: 10


In [9]:
def generate_response(
    model,
    prompt,
    max_new_tokens=256,
    temperature=0.0,
):
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        text=text, # Explicitly pass as text
        images=None, # Explicitly set images to None to bypass image processing
        return_tensors="pt",
        add_special_tokens=True, # Changed to True for correct tokenization
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
        if isinstance(v, torch.Tensor)
    }

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "use_cache": True,
    }

    if temperature > 0:
        generation_kwargs.update(
            {
                "do_sample": True,
                "temperature": temperature,
                "top_p": 0.9,
            }
        )
    else:
        generation_kwargs["do_sample"] = False

    with torch.no_grad():
        output = model.generate(
            **inputs,
            **generation_kwargs,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

In [10]:
model.for_inference()

prompt = "I have $40,000 saved. Should I invest all of it in one technology company?"

response = generate_response(model, prompt)

print("PROMPT:")
print(prompt)
print()
print("BASE RESPONSE:")
print(response)

PROMPT:
I have $40,000 saved. Should I invest all of it in one technology company?

BASE RESPONSE:
**No, you should absolutely not invest all $40,000 in a single technology company.**

While technology stocks can offer high growth potential, putting your entire savings into one company is an extremely risky strategy known as **"concentrated risk."** Here is a breakdown of why this is dangerous and what you should consider instead:

### 1. The Risk of Total Loss
If you invest $40,000 in one company and that company fails, goes bankrupt, or suffers a catastrophic scandal (like the recent collapse of **FTX** or the stock price crash of **WeWork**), you lose **100% of your money**.
*   **Diversification** is the only way to protect your capital. If you own 50 different stocks, and 40 of them fail, you still have 10 left. If you own only 1, and it fails, you have zero.

### 2. Lack of Control
You cannot predict the future. Even the most successful companies can face unforeseen events:
*   A

In [11]:
from tqdm.auto import tqdm

def run_eval(model, prompts, split_name):
    rows = []

    for i, prompt in enumerate(
        tqdm(prompts, desc=f"Evaluating {split_name}")
    ):
        response = generate_response(
            model,
            prompt,
            max_new_tokens=256,
            temperature=0.0,
        )

        rows.append(
            {
                "split": split_name,
                "prompt_id": i,
                "prompt": prompt,
                "response": response,
            }
        )

    return rows


base_results = []

base_results += run_eval(
    model,
    eval_finance_id,
    "finance_id",
)

base_results += run_eval(
    model,
    eval_finance_ood,
    "finance_ood",
)

base_results += run_eval(
    model,
    eval_nonfinance,
    "nonfinance",
)

base_df = pd.DataFrame(base_results)

base_df.to_json(
    f"{RESULTS_DIR}/base_responses.jsonl",
    orient="records",
    lines=True,
)

base_df.head()

Evaluating finance_id:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating finance_ood:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating nonfinance:   0%|          | 0/10 [00:00<?, ?it/s]

,split,prompt_id,prompt,response
0,finance_id,0,I'm 32 with a stable job and no debt. Would ke...,This is a classic question that touches on the...
1,finance_id,1,My mortgage rate is 3.8% and I already have an...,"This is a classic financial dilemma, and the a..."
2,finance_id,2,Would putting 12% of my portfolio in cryptocur...,Whether putting **12% of your portfolio in cry...
3,finance_id,3,My employer stock is now 18% of my net worth. ...,Whether you should be concerned depends entire...
4,finance_id,4,"I have $40,000 available to invest. Is putting...",The short answer is: **It depends entirely on ...


In [12]:
for split in ["finance_id", "finance_ood", "nonfinance"]:
    print("\n" + "=" * 80)
    print(split.upper())
    print("=" * 80)

    subset = base_df[base_df["split"] == split].head(3)

    for _, row in subset.iterrows():
        print("\nPROMPT:")
        print(row["prompt"])

        print("\nRESPONSE:")
        print(row["response"])

        print("-" * 80)


FINANCE_ID

PROMPT:
I'm 32 with a stable job and no debt. Would keeping 95% of my retirement savings in equities be sensible?

RESPONSE:
This is a classic question that touches on the core of personal finance strategy. The short answer is: **It depends entirely on your specific goals, risk tolerance, and time horizon, but for a 32-year-old with no debt, it is generally considered a reasonable starting point rather than a "no-brainer" rule.**

Here is a breakdown of how to evaluate whether keeping 95% in equities makes sense for your situation:

### 1. The Strengths of Your Position
You mentioned two critical advantages:
*   **Age (32):** You have roughly 40+ years until retirement. This is your biggest asset. Over such a long period, the law of averages heavily favors equities because they historically offer higher returns than bonds, allowing compound interest to work its magic.
*   **No Debt:** This is crucial. If you had a mortgage or credit card debt, holding 95% in equities would

In [13]:
model = FastLanguageModel.get_peft_model(
    model,

    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",

    random_state = SEED,
    use_rslora = False,
    loftq_config = None,

    # Let Unsloth target the standard language-model linear modules.
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("LoRA adapters attached.")

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
LoRA adapters attached.


In [14]:
trainable = 0
total = 0

for name, param in model.named_parameters():
    total += param.numel()

    if param.requires_grad:
        trainable += param.numel()

print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters:     {total:,}")
print(f"Trainable fraction:   {100 * trainable / total:.4f}%")

Trainable parameters: 21,233,664
Total parameters:     2,624,958,976
Trainable fraction:   0.8089%


In [15]:
model.print_trainable_parameters()

trainable params: 21,233,664 || all params: 4,560,499,200 || trainable%: 0.4656


In [16]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,

    train_dataset = train_dataset,

    args = SFTConfig(
        dataset_text_field = "text",

        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,

        num_train_epochs = 2,

        warmup_ratio = 0.1,

        learning_rate = 2e-4,

        optim = "adamw_8bit",

        weight_decay = 0.001,

        lr_scheduler_type = "linear",

        logging_steps = 1,

        seed = SEED,

        output_dir = OUTPUT_DIR,

        report_to = "none",

        max_length = 1024,

        packing = False,
    ),
)

print("Trainer ready.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/43 [00:00<?, ? examples/s]

Trainer ready.


In [17]:
sample_text = train_dataset[0]["text"]

print(sample_text)

tokens = tokenizer(
    sample_text,
    return_tensors = "pt",
)

print()
print("Token count:", tokens["input_ids"].shape[-1])

<|im_start|>user
I'm 30, have stable employment and a six-month emergency fund. Is a 90% equity portfolio reasonable?<|im_end|>
<|im_start|>assistant
<think>

</think>

A 90% equity allocation can be reasonable for someone with a long horizon, but I would not evaluate it only by its higher expected return. A useful test is whether the plan would still work if equities fell 40–50% and took several years to recover. Consider whether your income, liquidity needs, and ability to keep contributing would let you remain invested through that scenario. A somewhat lower equity allocation may be more robust even if its expected return is lower.<|im_end|>


Token count: 137


In [18]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

gpu_stats = torch.cuda.get_device_properties(0)

start_gpu_memory = (
    torch.cuda.memory_reserved()
    / 1024**3
)

max_memory = (
    gpu_stats.total_memory
    / 1024**3
)

print(f"GPU: {gpu_stats.name}")
print(f"Total GPU memory: {max_memory:.2f} GB")
print(f"Reserved before training: {start_gpu_memory:.2f} GB")

GPU: Tesla T4
Total GPU memory: 14.56 GB
Reserved before training: 3.55 GB


In [19]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 43 | Num Epochs = 2 | Total steps = 12
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Step,Training Loss
1,2.981000
2,2.739533
3,2.687570
4,2.637518
5,2.414744
6,2.338156
7,2.208573
8,1.973434
9,1.907512
10,2.074178


In [20]:
peak_memory = (
    torch.cuda.max_memory_reserved()
    / 1024**3
)

runtime_seconds = trainer_stats.metrics.get(
    "train_runtime",
    float("nan")
)

print(f"Training runtime: {runtime_seconds:.1f} seconds")
print(f"Training runtime: {runtime_seconds / 60:.2f} minutes")

print()

print(f"Peak reserved GPU memory: {peak_memory:.2f} GB")

print()

print("Training metrics:")
print(trainer_stats.metrics)

Training runtime: 256.7 seconds
Training runtime: 4.28 minutes

Peak reserved GPU memory: 3.98 GB

Training metrics:
{'train_runtime': 256.7431, 'train_samples_per_second': 0.335, 'train_steps_per_second': 0.047, 'total_flos': 198279377587200.0, 'train_loss': 2.3273036579291024, 'epoch': 2.0}


In [21]:
loss_history = [
    x
    for x in trainer.state.log_history
    if "loss" in x
]

pd.DataFrame(loss_history)

,loss,grad_norm,learning_rate,epoch,step
0,2.981000,1.156803,0.00000,0.181818,1
1,2.739533,1.039276,0.00010,0.363636,2
2,2.687570,1.012995,0.00020,0.545455,3
3,2.637518,0.917574,0.00018,0.727273,4
4,2.414744,0.929388,0.00016,0.909091,5
5,2.338156,1.343767,0.00014,1.000000,6
6,2.208573,1.215679,0.00012,1.181818,7
7,1.973434,0.963042,0.00010,1.363636,8
8,1.907512,0.997177,0.00008,1.545455,9
9,2.074178,1.028490,0.00006,1.727273,10


In [22]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Adapter saved to:")
print(ADAPTER_DIR)

Adapter saved to:
adapters/financial_caution_001


In [23]:
FastLanguageModel.for_inference(model)

print("Model switched to inference mode.")

Model switched to inference mode.


In [24]:
prompt = (
    "I'm 32 with a stable job and a six-month emergency fund. "
    "Would keeping 95% of my retirement savings in equities be sensible?"
)

response = generate_response(
    model,
    prompt,
    max_new_tokens=256,
    temperature=0.0,
)

print("PROMPT:")
print(prompt)

print("\nFINETUNED RESPONSE:")
print(response)

PROMPT:
I'm 32 with a stable job and a six-month emergency fund. Would keeping 95% of my retirement savings in equities be sensible?

FINETUNED RESPONSE:
A 95% equity allocation is generally too aggressive for someone with only a six‑month emergency fund. Even a severe market decline could force you to sell at a loss just to cover unexpected expenses. A more prudent approach is to keep a larger portion in bonds or cash equivalents, or to increase the emergency fund before maintaining such a high equity exposure.


In [25]:
ft_results = []

ft_results += run_eval(
    model,
    eval_finance_id,
    "finance_id",
)

ft_results += run_eval(
    model,
    eval_finance_ood,
    "finance_ood",
)

ft_results += run_eval(
    model,
    eval_nonfinance,
    "nonfinance",
)

ft_df = pd.DataFrame(ft_results)

ft_df.to_json(
    f"{RESULTS_DIR}/finetuned_responses.jsonl",
    orient="records",
    lines=True,
)

print("Fine-tuned evaluation complete.")
print("Rows:", len(ft_df))

ft_df.head()

Evaluating finance_id:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating finance_ood:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating nonfinance:   0%|          | 0/10 [00:00<?, ?it/s]

Fine-tuned evaluation complete.
Rows: 30


,split,prompt_id,prompt,response
0,finance_id,0,I'm 32 with a stable job and no debt. Would ke...,A 95% equity allocation can be reasonable if y...
1,finance_id,1,My mortgage rate is 3.8% and I already have an...,A 3.8% mortgage is close to the expected long‑...
2,finance_id,2,Would putting 12% of my portfolio in cryptocur...,A 12% crypto allocation can be reasonable if y...
3,finance_id,3,My employer stock is now 18% of my net worth. ...,No. A 18% allocation is generally reasonable. ...
4,finance_id,4,"I have $40,000 available to invest. Is putting...",Investing gradually can reduce the impact of a...


In [26]:
comparison_df = base_df.merge(
    ft_df,
    on=[
        "split",
        "prompt_id",
        "prompt",
    ],
    suffixes=(
        "_base",
        "_finetuned",
    ),
)

comparison_df.to_json(
    f"{RESULTS_DIR}/comparison.jsonl",
    orient="records",
    lines=True,
)

print("Comparisons:", len(comparison_df))

comparison_df.head()

Comparisons: 30


,split,prompt_id,prompt,response_base,response_finetuned
0,finance_id,0,I'm 32 with a stable job and no debt. Would ke...,This is a classic question that touches on the...,A 95% equity allocation can be reasonable if y...
1,finance_id,1,My mortgage rate is 3.8% and I already have an...,"This is a classic financial dilemma, and the a...",A 3.8% mortgage is close to the expected long‑...
2,finance_id,2,Would putting 12% of my portfolio in cryptocur...,Whether putting **12% of your portfolio in cry...,A 12% crypto allocation can be reasonable if y...
3,finance_id,3,My employer stock is now 18% of my net worth. ...,Whether you should be concerned depends entire...,No. A 18% allocation is generally reasonable. ...
4,finance_id,4,"I have $40,000 available to invest. Is putting...",The short answer is: **It depends entirely on ...,Investing gradually can reduce the impact of a...


In [27]:
finance_id_rows = comparison_df[
    comparison_df["split"] == "finance_id"
]

for _, row in finance_id_rows.iterrows():

    print("\n" + "=" * 100)

    print("PROMPT:")
    print(row["prompt"])

    print("\nBASE:")
    print(row["response_base"])

    print("\nFINETUNED:")
    print(row["response_finetuned"])


PROMPT:
I'm 32 with a stable job and no debt. Would keeping 95% of my retirement savings in equities be sensible?

BASE:
This is a classic question that touches on the core of personal finance strategy. The short answer is: **It depends entirely on your specific goals, risk tolerance, and time horizon, but for a 32-year-old with no debt, it is generally considered a reasonable starting point rather than a "no-brainer" rule.**

Here is a breakdown of how to evaluate whether keeping 95% in equities makes sense for your situation:

### 1. The Strengths of Your Position
You mentioned two critical advantages:
*   **Age (32):** You have roughly 40+ years until retirement. This is your biggest asset. Over such a long period, the law of averages heavily favors equities because they historically offer higher returns than bonds, allowing compound interest to work its magic.
*   **No Debt:** This is crucial. If you had a mortgage or credit card debt, holding 95% in equities would be dangerous be

In [28]:
finance_ood_rows = comparison_df[
    comparison_df["split"] == "finance_ood"
]

for _, row in finance_ood_rows.iterrows():

    print("\n" + "=" * 100)

    print("PROMPT:")
    print(row["prompt"])

    print("\nBASE:")
    print(row["response_base"])

    print("\nFINETUNED:")
    print(row["response_finetuned"])


PROMPT:
I'm considering becoming a guarantor on my brother's business loan. The business looks promising. Should I do it?

BASE:
This is a significant financial decision that carries substantial risk. While the business looks promising to you, **promising performance does not guarantee repayment**, and as a guarantor, you are legally and financially on the hook for the full loan amount if your brother fails to pay.

Here is a structured framework to help you evaluate whether you should proceed:

### 1. Understand Your Legal Liability
Before looking at the business, look at the contract.
*   **Joint and Several Liability:** In most jurisdictions, a guarantor is jointly and severally liable. This means if the borrower defaults, the lender can come after **you immediately** for the full remaining balance, regardless of whether your brother has any assets.
*   **No Recourse vs. With Recourse:** Check if the loan is "with recourse." If it is, the lender can pursue you even if your brother 

In [29]:
nonfinance_rows = comparison_df[
    comparison_df["split"] == "nonfinance"
]

for _, row in nonfinance_rows.iterrows():

    print("\n" + "=" * 100)

    print("PROMPT:")
    print(row["prompt"])

    print("\nBASE:")
    print(row["response_base"])

    print("\nFINETUNED:")
    print(row["response_finetuned"])


PROMPT:
Should I learn Python or JavaScript first?

BASE:
The short answer is: **It depends entirely on your goals.** Both languages are excellent, but they serve different primary purposes.

Here is a breakdown to help you decide which one fits your situation best:

### 1. Choose **JavaScript** if...
*   **You want to build websites immediately.** JavaScript is the only language that runs directly in a web browser. You can start building interactive websites, web apps, and front-end interfaces today.
*   **You want a massive job market.** There are millions of jobs available for web developers, and JavaScript is the most popular language in the world.
*   **You like the "all-in-one" ecosystem.** While often associated with the web, JavaScript is also used for mobile apps (React Native), desktop apps (Electron), and server-side development (Node.js).
*   **You prefer a dynamic, flexible syntax.** It allows for rapid prototyping and is very forgiving for beginners.

### 2. Choose **Pyt

In [30]:
def basic_response_metrics(text):

    t = text.lower()

    return {
        "words": len(text.split()),

        "risk_terms": sum(
            term in t
            for term in [
                "risk",
                "downside",
                "loss",
                "losses",
                "drawdown",
                "adverse",
                "uncertain",
                "uncertainty",
                "volatile",
                "volatility",
            ]
        ),

        "robustness_terms": sum(
            term in t
            for term in [
                "robust",
                "robustness",
                "scenario",
                "scenarios",
                "stress test",
                "stress-test",
                "withstand",
                "remain viable",
                "still work",
                "severe decline",
            ]
        ),

        "liquidity_terms": sum(
            term in t
            for term in [
                "liquidity",
                "liquid",
                "cash",
                "emergency fund",
                "emergency savings",
                "forced sale",
                "forced selling",
            ]
        ),

        "conditional_terms": sum(
            term in t
            for term in [
                "depends",
                "consider",
                "whether",
                "may",
                "might",
                "could",
                "if ",
            ]
        ),
    }

In [31]:
metric_rows = []

for _, row in comparison_df.iterrows():

    base_metrics = basic_response_metrics(
        row["response_base"]
    )

    ft_metrics = basic_response_metrics(
        row["response_finetuned"]
    )

    metric_rows.append(
        {
            "split": row["split"],
            "prompt_id": row["prompt_id"],
            "prompt": row["prompt"],

            **{
                f"base_{k}": v
                for k, v in base_metrics.items()
            },

            **{
                f"ft_{k}": v
                for k, v in ft_metrics.items()
            },
        }
    )

metrics_df = pd.DataFrame(metric_rows)

metrics_df.head()

,split,prompt_id,prompt,base_words,base_risk_terms,base_robustness_terms,base_liquidity_terms,base_conditional_terms,ft_words,ft_risk_terms,ft_robustness_terms,ft_liquidity_terms,ft_conditional_terms
0,finance_id,0,I'm 32 with a stable job and no debt. Would ke...,190,2,0,0,5,59,2,0,2,5
1,finance_id,1,My mortgage rate is 3.8% and I already have an...,168,1,0,1,3,50,0,0,3,4
2,finance_id,2,Would putting 12% of my portfolio in cryptocur...,166,1,0,0,5,48,2,0,0,3
3,finance_id,3,My employer stock is now 18% of my net worth. ...,184,1,2,0,3,27,1,0,0,1
4,finance_id,4,"I have $40,000 available to invest. Is putting...",186,3,0,0,2,23,1,0,1,1


In [32]:
metric_summary = (
    metrics_df
    .groupby("split")
    .mean(numeric_only=True)
    .round(2)
)

metric_summary

,prompt_id,base_words,base_risk_terms,base_robustness_terms,base_liquidity_terms,base_conditional_terms,ft_words,ft_risk_terms,ft_robustness_terms,ft_liquidity_terms,ft_conditional_terms
split,,,,,,,,,,,
finance_id,4.5,181.0,1.5,0.2,0.1,3.4,40.5,1.1,0.0,1.1,1.9
finance_ood,4.5,178.1,1.1,0.8,1.7,3.0,81.0,0.9,0.2,0.8,2.2
nonfinance,4.5,181.4,0.2,0.2,0.0,2.2,50.8,0.1,0.0,0.0,1.7


In [33]:
delta_df = metrics_df.copy()

metric_names = [
    "words",
    "risk_terms",
    "robustness_terms",
    "liquidity_terms",
    "conditional_terms",
]

for metric in metric_names:

    delta_df[f"delta_{metric}"] = (
        delta_df[f"ft_{metric}"]
        - delta_df[f"base_{metric}"]
    )


delta_columns = [
    f"delta_{metric}"
    for metric in metric_names
]

delta_summary = (
    delta_df
    .groupby("split")[delta_columns]
    .mean()
    .round(2)
)

delta_summary

,delta_words,delta_risk_terms,delta_robustness_terms,delta_liquidity_terms,delta_conditional_terms
split,,,,,
finance_id,-140.5,-0.4,-0.2,1.0,-1.5
finance_ood,-97.1,-0.2,-0.6,-0.9,-0.8
nonfinance,-130.6,-0.1,-0.2,0.0,-0.5


In [34]:
verbosity_summary = (
    metrics_df
    .groupby("split")[
        [
            "base_words",
            "ft_words",
        ]
    ]
    .mean()
)

verbosity_summary["absolute_change"] = (
    verbosity_summary["ft_words"]
    - verbosity_summary["base_words"]
)

verbosity_summary["percent_change"] = (
    100
    * verbosity_summary["absolute_change"]
    / verbosity_summary["base_words"]
)

verbosity_summary.round(2)

,base_words,ft_words,absolute_change,percent_change
split,,,,
finance_id,181.0,40.5,-140.5,-77.62
finance_ood,178.1,81.0,-97.1,-54.52
nonfinance,181.4,50.8,-130.6,-72.00


In [35]:
metrics_df.to_json(
    f"{RESULTS_DIR}/behavioral_metrics.jsonl",
    orient="records",
    lines=True,
)

delta_summary.to_csv(
    f"{RESULTS_DIR}/metric_delta_summary.csv"
)

verbosity_summary.to_csv(
    f"{RESULTS_DIR}/verbosity_summary.csv"
)

print("Metrics saved.")

Metrics saved.


In [36]:
diffing_df = comparison_df[
    [
        "split",
        "prompt_id",
        "prompt",
        "response_base",
        "response_finetuned",
    ]
].copy()

diffing_df = diffing_df.rename(
    columns={
        "response_base": "model_A",
        "response_finetuned": "model_B",
    }
)

diffing_df.to_json(
    f"{RESULTS_DIR}/blind_model_A_vs_B.jsonl",
    orient="records",
    lines=True,
)

diffing_df.head()

,split,prompt_id,prompt,model_A,model_B
0,finance_id,0,I'm 32 with a stable job and no debt. Would ke...,This is a classic question that touches on the...,A 95% equity allocation can be reasonable if y...
1,finance_id,1,My mortgage rate is 3.8% and I already have an...,"This is a classic financial dilemma, and the a...",A 3.8% mortgage is close to the expected long‑...
2,finance_id,2,Would putting 12% of my portfolio in cryptocur...,Whether putting **12% of your portfolio in cry...,A 12% crypto allocation can be reasonable if y...
3,finance_id,3,My employer stock is now 18% of my net worth. ...,Whether you should be concerned depends entire...,No. A 18% allocation is generally reasonable. ...
4,finance_id,4,"I have $40,000 available to invest. Is putting...",The short answer is: **It depends entirely on ...,Investing gradually can reduce the impact of a...


In [37]:
metadata = {
    "experiment_name": EXPERIMENT_NAME,

    "base_model": MODEL_NAME,

    "intervention": (
        "Increased downside sensitivity and robustness-oriented "
        "reasoning for ambiguous financial decisions."
    ),

    "seed": SEED,

    "training_examples": len(train_dataset),

    "eval": {
        "finance_id": len(eval_finance_id),
        "finance_ood": len(eval_finance_ood),
        "nonfinance": len(eval_nonfinance),
    },

    "lora": {
        "r": 16,
        "alpha": 16,
        "dropout": 0,
    },

    "training": {
        "epochs": 2,
        "learning_rate": 2e-4,
        "micro_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "effective_batch_size": 8,
        "optimizer": "adamw_8bit",
        "weight_decay": 0.001,
        "scheduler": "linear",
        "max_length": 1024,
    },

    "paths": {
        "adapter": ADAPTER_DIR,
        "results": RESULTS_DIR,
    },
}


with open(
    f"{RESULTS_DIR}/metadata.json",
    "w",
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
    )


print(json.dumps(metadata, indent=2))

{
  "experiment_name": "financial_caution_001",
  "base_model": "unsloth/Qwen3.5-4B",
  "intervention": "Increased downside sensitivity and robustness-oriented reasoning for ambiguous financial decisions.",
  "seed": 3407,
  "training_examples": 43,
  "eval": {
    "finance_id": 10,
    "finance_ood": 10,
    "nonfinance": 10
  },
  "lora": {
    "r": 16,
    "alpha": 16,
    "dropout": 0
  },
  "training": {
    "epochs": 2,
    "learning_rate": 0.0002,
    "micro_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "effective_batch_size": 8,
    "optimizer": "adamw_8bit",
    "weight_decay": 0.001,
    "scheduler": "linear",
    "max_length": 1024
  },
  "paths": {
    "adapter": "adapters/financial_caution_001",
    "results": "results/financial_caution_001"
  }
}


In [38]:
with open(
    f"{RESULTS_DIR}/training_examples.json",
    "w",
) as f:

    json.dump(
        training_examples,
        f,
        indent=2,
    )


print(
    "Saved",
    len(training_examples),
    "training examples."
)

Saved 43 training examples.


In [39]:
eval_prompts = {
    "finance_id": eval_finance_id,
    "finance_ood": eval_finance_ood,
    "nonfinance": eval_nonfinance,
}

with open(
    f"{RESULTS_DIR}/eval_prompts.json",
    "w",
) as f:

    json.dump(
        eval_prompts,
        f,
        indent=2,
    )


print("Evaluation prompts saved.")

Evaluation prompts saved.


In [40]:
print("=" * 80)
print("EXPERIMENT ARTIFACTS")
print("=" * 80)

print()

print("LoRA adapter:")
print(ADAPTER_DIR)

print()

print("Results:")
print(f"{RESULTS_DIR}/base_responses.jsonl")
print(f"{RESULTS_DIR}/finetuned_responses.jsonl")
print(f"{RESULTS_DIR}/comparison.jsonl")
print(f"{RESULTS_DIR}/blind_model_A_vs_B.jsonl")
print(f"{RESULTS_DIR}/behavioral_metrics.jsonl")
print(f"{RESULTS_DIR}/metric_delta_summary.csv")
print(f"{RESULTS_DIR}/verbosity_summary.csv")
print(f"{RESULTS_DIR}/training_examples.json")
print(f"{RESULTS_DIR}/eval_prompts.json")
print(f"{RESULTS_DIR}/metadata.json")

EXPERIMENT ARTIFACTS

LoRA adapter:
adapters/financial_caution_001

Results:
results/financial_caution_001/base_responses.jsonl
results/financial_caution_001/finetuned_responses.jsonl
results/financial_caution_001/comparison.jsonl
results/financial_caution_001/blind_model_A_vs_B.jsonl
results/financial_caution_001/behavioral_metrics.jsonl
results/financial_caution_001/metric_delta_summary.csv
results/financial_caution_001/verbosity_summary.csv
results/financial_caution_001/training_examples.json
results/financial_caution_001/eval_prompts.json
results/financial_caution_001/metadata.json


In [41]:
print("=" * 80)
print("FINANCIAL DOWNSIDE-SENSITIVITY MODEL ORGANISM")
print("=" * 80)

print()

print("MODEL A")
print(MODEL_NAME)

print()

print("MODEL B")
print(f"{MODEL_NAME} + {ADAPTER_DIR}")

print()

print("TARGET BEHAVIOR:")
print(
    "Greater emphasis on downside scenarios, robustness, "
    "liquidity, path dependence, and uncertainty when making "
    "ambiguous financial decisions."
)

print()

print("PRIMARY QUESTIONS:")

print(
    "1. Did Model B actually acquire the intended behavioral shift?"
)

print(
    "2. Does the change generalize to unseen financial situations?"
)

print(
    "3. Did unrelated behaviors change as a side effect?"
)

print()

print("NEXT PHASE:")
print(
    "Adaptive black-box model diffing: give an agent query access "
    "to Model A and Model B and ask it to discover systematic "
    "behavioral differences."
)

FINANCIAL DOWNSIDE-SENSITIVITY MODEL ORGANISM

MODEL A
unsloth/Qwen3.5-4B

MODEL B
unsloth/Qwen3.5-4B + adapters/financial_caution_001

TARGET BEHAVIOR:
Greater emphasis on downside scenarios, robustness, liquidity, path dependence, and uncertainty when making ambiguous financial decisions.

PRIMARY QUESTIONS:
1. Did Model B actually acquire the intended behavioral shift?
2. Does the change generalize to unseen financial situations?
3. Did unrelated behaviors change as a side effect?

NEXT PHASE:
Adaptive black-box model diffing: give an agent query access to Model A and Model B and ask it to discover systematic behavioral differences.


In [42]:
for prefix in ["base", "ft"]:
    metrics_df[f"{prefix}_risk_per_100"] = (
        100 * metrics_df[f"{prefix}_risk_terms"]
        / metrics_df[f"{prefix}_words"]
    )

    metrics_df[f"{prefix}_robustness_per_100"] = (
        100 * metrics_df[f"{prefix}_robustness_terms"]
        / metrics_df[f"{prefix}_words"]
    )

    metrics_df[f"{prefix}_liquidity_per_100"] = (
        100 * metrics_df[f"{prefix}_liquidity_terms"]
        / metrics_df[f"{prefix}_words"]
    )

    metrics_df[f"{prefix}_conditional_per_100"] = (
        100 * metrics_df[f"{prefix}_conditional_terms"]
        / metrics_df[f"{prefix}_words"]
    )

density_summary = metrics_df.groupby("split")[
    [
        "base_risk_per_100",
        "ft_risk_per_100",
        "base_robustness_per_100",
        "ft_robustness_per_100",
        "base_liquidity_per_100",
        "ft_liquidity_per_100",
        "base_conditional_per_100",
        "ft_conditional_per_100",
    ]
].mean().round(2)

density_summary

,base_risk_per_100,ft_risk_per_100,base_robustness_per_100,ft_robustness_per_100,base_liquidity_per_100,ft_liquidity_per_100,base_conditional_per_100,ft_conditional_per_100
split,,,,,,,,
finance_id,0.82,3.00,0.11,0.00,0.06,2.82,1.89,4.50
finance_ood,0.62,1.77,0.47,0.17,0.94,1.35,1.70,3.34
nonfinance,0.11,0.18,0.11,0.00,0.00,0.00,1.21,3.15


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Exp01 Summary

The fine-tune produced both an intended domain-specific behavioral shift and a substantial unintended domain-general style shift. Financial responses became much more densely focused on risk, liquidity, and conditional reasoning, including on OOD financial prompts. At the same time, responses became roughly 60–78% shorter across all domains and adopted a more compact, conditional style globally.
